In [ ]:
# Install notebook dependencies
%pip install pandas
%pip install matplotlib
%pip install numpy==1.25.0
%pip install seaborn
%pip install palettable

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import seaborn as sns
import matplotlib as mpl
import palettable

#plt.style.use(['dark_background'])
plt.rcParams['figure.figsize'] = [15, 9]
# Set general font size
plt.rcParams['font.size'] = '24'
plt.rcParams['savefig.dpi'] = 200
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
plt_colors = ['cornflowerblue', 'crimson', 'forestgreen', 'mediumorchid']
plt.style.use('seaborn-v0_8-colorblind')

sns.set_theme(style='ticks')
sns.set_context("poster")
plt.rc('font', family='serif')

cycler = mpl.cycler(color=palettable.scientific.sequential.Acton_7.mpl_colors, marker=['o', '+', 'x', '*', '.', 'X', 'd'])
hatches = ['/', '//', '|', '+', 'x', 'o', '-']

mpl.rcParams['axes.prop_cycle'] = cycler

def set_hatches(ax:plt.Axes, df:pd.DataFrame):
    bars = ax.patches
    hatches = ''.join(h*len(df) for h in '/+xo-|')
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)

## Primary results -- active vs passive synchronization

In [ ]:
f = 'active_passive_synchronization_zxxz.pkl'

with open(f, 'rb') as file:
    data = pickle.load(file)

f = 'active_passive_synchronization_zzxx.pkl'

with open(f, 'rb') as file:
    data_zzxx = pickle.load(file)


fs = 48
conv = lambda l:r'$\tau$: %d ns'%(l)
ds = range(3, 16, 2)
ts = [500, 1000]

for ls in ['Z', 'X']:
    for i in range(2):
        fig, axs = plt.subplots(1, 2, constrained_layout=True, figsize=(30, 10))
        ax = axs[0]
        sync = {conv(i):{} for i in ts}
        basis = 'Z'
        if basis == ls:
            dataset = data_zzxx
            title = r'$X_{P}X_{P}$' + '\''
        else:
            dataset = data
            title = r'$Z_{P}$'
        for d in ds:
            for t in ts:
                sync[conv(t)][d] = (1 - dataset[i][((1, basis, ls, d, t))] / dataset[i][((0, basis, ls, d, t))]) * 100
                sync[conv(t)][d] = 100 / (100 - sync[conv(t)][d])
                pass
        df = pd.DataFrame(sync)
        df.plot(ax=ax, xticks=range(3, 16, 2), markersize=28, legend=False)
        ax.grid(axis='y', linestyle='--')
        ax.set_ylabel(r'Reduction', math_fontfamily='dejavuserif', fontsize=fs)
        ax.set_xlabel('Code Distance (d)', fontsize=fs)
        # ax.legend(fontsize=fs-6, ncol=2, fancybox=False, shadow=True)
        ax.set_title(title, fontsize=fs+12)
        plt.setp(ax.get_xticklabels(), fontsize=fs)
        plt.setp(ax.get_yticklabels(), fontsize=fs)
        ax.axhline(y=1, linestyle='--', color='k')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        ax = axs[1]
        sync = {conv(i):{} for i in ts}
        basis = 'X'
        if basis == ls:
            dataset = data_zzxx
            title = r'$Z_{P}Z_{P}$' + '\''
        else:
            dataset = data
            title = r'$X_{P}$'
        for d in ds:
            for t in ts:
                sync[conv(t)][d] = (1 - dataset[i][((1, basis, ls, d, t))] / dataset[i][((0, basis, ls, d, t))]) * 100
                sync[conv(t)][d] = 100 / (100 - sync[conv(t)][d])
                pass
        df = pd.DataFrame(sync)
        df.plot(ax=ax, xticks=range(3, 16, 2), markersize=28, legend=False)
        ax.grid(axis='y', linestyle='--')
        ax.set_ylabel(r'Reduction', math_fontfamily='dejavuserif', fontsize=fs)
        ax.set_xlabel('Code Distance (d)', fontsize=fs)
        # ax.legend(fontsize=fs-6, ncol=2, fancybox=False, shadow=True)
        ax.set_title(title, fontsize=fs + 12)
        plt.setp(ax.get_xticklabels(), fontsize=fs)
        plt.setp(ax.get_yticklabels(), fontsize=fs)
        ax.axhline(y=1, linestyle='--', color='k')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        handles, labels = ax.get_legend_handles_labels()
        handle_list, label_list = [], []
        for handle, label in zip(handles, labels):
            if label not in label_list:
                handle_list.append(handle)
                label_list.append(label)
        lgd = fig.legend(handle_list, label_list, ncol=4, loc='lower center', fancybox=False, shadow=True, fontsize=fs, bbox_to_anchor=(0.5,-0.16))
        mysup = fig.suptitle('%s-basis Lattice Surgery'%(ls), fontsize=fs)

## Gap in LER between ideal case and active/passive synchronization

In [ ]:
f = 'active_passive_synchronization_zxxz.pkl'
fs=48
with open(f, 'rb') as file:
    data_old = pickle.load(file)

with open('./active_passive_synchronization_ideal_zx.pkl', 'rb') as file:
    ideal = pickle.load(file)

with open('./active_passive_synchronization_ideal_zzxx.pkl', 'rb') as file:
    ideal_zz = pickle.load(file)

f = 'active_passive_synchronization_zzxx.pkl'

with open(f, 'rb') as file:
    data_xxzz = pickle.load(file)

ds = range(3, 16, 2)
fig, axs = plt.subplots(1, 2, constrained_layout=True, figsize=(30, 10))
ls = 'Z'
lers = {}
t = 1000
for i, basis in enumerate(['Z','X']):
    ax = axs[i]
    if ls == basis:
        title = r'$X_{P}X_{P}$' + '\''
        data = data_xxzz
        data_ideal = ideal_zz
    else:
        title = r'$X_{P}$'
        data = data_old
        data_ideal = ideal
    lers['Ideal'] = {d:data_ideal[0][((2, basis, ls, d, t))] for d in range(3, 16, 2)}
    lers['Active'] = {d:data[0][((1, basis, ls, d, t))] for d in range(3, 16, 2)}
    lers['Passive'] = {d:data[0][((0, basis, ls, d, t))] for d in range(3, 16, 2)}
    df = pd.DataFrame(lers)
    df.plot(rot=0, ax=ax, markersize=28)
    ax.grid(axis='y', linestyle='--')
    ax.set_ylabel(r'Logical Error Rate', math_fontfamily='dejavuserif', fontsize=fs)
    ax.set_xlabel('Code Distance (d)', fontsize=fs)
    ax.legend(fontsize=fs, ncol=1, fancybox=False, shadow=True)
    ax.set_title(title, fontsize=fs+12)
    ax.set_yscale('log')
    ax.set_xticks(range(3, 16, 2))
    plt.setp(ax.get_xticklabels(), fontsize=fs)
    plt.setp(ax.get_yticklabels(), fontsize=fs)
    ax.axhline(y=0, linestyle='--', color='k')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    pass


## Running active synchronization with additional rounds

In [ ]:
with open('active_passive_synchronization_rounds.pkl', 'rb') as file:
    results = pickle.load(file)

conv = lambda l:r'$\tau$: %d ns'%(l)
results = results[0]
rounds = range(11)
active = {}
slacks = [500, 1000]
for slack in slacks:
    active[conv(slack)] = {}
    for round in rounds:
        p = np.mean([val for key, val in results.items() if key[0] == 0 and key[-1] == round and key[-2] == slack])
        active[conv(slack)][round] = p / np.mean([val for key, val in results.items() if key[0] == 1 and key[-1] == round and key[-2] == slack])
        pass

fs=44
fig, ax = plt.subplots(1, 1, constrained_layout=True, figsize=(12, 7))
df = pd.DataFrame(active)
df.plot(rot=0, ax=ax, markersize=30)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.setp(ax.get_xticklabels(), fontsize=fs)
plt.setp(ax.get_yticklabels(), fontsize=fs)
ax.set_ylabel('Reduction', fontsize=fs)
ax.set_xlabel('Additional Rounds (R)', fontsize=fs)
ax.grid(linestyle='--', axis='y')
ax.axhline(y = 1, linestyle='--', color='k')
ax.set_title(r'$d=11$', fontsize=fs)
ax.legend(ncol=1, fontsize=fs)

## Hybrid synchronization

In [ ]:
with open('hybrid_sync.pkl', 'rb') as file:
    results = pickle.load(file)

epsilons = [100, 200, 300, 400]
slacks = [500, 1000]
modes = {'Active':1, 'Extra\nRounds':2}
hybrid_modes = {'Hybrid \n(' + r'$\epsilon:$' + ' %i)'%(i):i for i in epsilons}

ds = [11]

def sformat(s:int) -> str:
    return r'$\tau$: %d ns'%(s)
# active, p, ls_basis, d, idle, rounds, orig_slack, c, eps

proc_data = {}
rounds = {}
for d in ds:
    proc_data[d] = {}
    rounds[d] = {}
    errs = []
    for mode, i in modes.items():
        proc_data[d][mode] = {}
        rounds[d][mode]= {}
        for slack in slacks:
            p = np.mean([val for key, val in results[0].items() if key[0] == 0 and key[-1] == 0 and key[3] == d and key[-3] == slack])
            errs = [val for key, val in results[0].items() if key[0] == i and key[-1] == 0 and key[3] == d and key[-3] == slack]
            proc_data[d][mode][sformat(slack)] = p / np.mean(errs)
            rounds[d][mode][slack] = np.mean([key[-4] for key in results[0].keys() if key[0] == i and key[-1] == 0 and key[3] == d and key[-3] == slack])
        pass
    for mode, i in hybrid_modes.items():
        proc_data[d][mode] = {}
        rounds[d][mode]= {}
        for slack in slacks:
            p = np.mean([val for key, val in results[0].items() if key[0] == 0 and key[-1] == 0 and key[3] == d and key[-3] == slack])
            errs = [val for key, val in results[0].items() if key[0] == 3 and key[-1] == i and key[3] == d and key[-3] == slack]
            proc_data[d][mode][sformat(slack)] = p / np.mean(errs)
            rounds[d][mode][slack] = np.mean([key[-4] for key in results[0].keys() if key[0] == 3 and key[-1] == i and key[3] == d and key[-3] == slack])
        pass

print(proc_data[11]['Active'])

fig, ax = plt.subplots(1, 1, constrained_layout=True, figsize=(20, 8))
df = pd.DataFrame(proc_data[11]).T
df.plot.bar(rot=45, ax=ax, legend=False, edgecolor='k')
set_hatches(ax, df)
# ax.set_yscale('log')
ax.grid(linestyle='--', axis='y')
fs = 44
ax.set_ylabel('Reduction', fontsize=fs)
# ax.set_xlabel(r'Initial Slack, $\tau$ (ns)', fontsize=fs)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.setp(ax.get_xticklabels(), fontsize=fs)
plt.setp(ax.get_yticklabels(), fontsize=fs)
# ax.legend(ncol=3, fontsize=fs)
ax.axhline(y=1, linestyle='--', color='k')
ax.set_title(r'$d=11$', fontsize=fs)
handles, labels = ax.get_legend_handles_labels()
handle_list, label_list = [], []
for handle, label in zip(handles, labels):
    if label not in label_list:
        handle_list.append(handle)
        label_list.append(label)
lgd = fig.legend(handle_list, label_list, ncol=3, loc='lower center', fancybox=False, shadow=True, fontsize=fs, bbox_to_anchor=(0.5,-0.2))

## Hybrid synchronization -- neutral atoms

In [ ]:
with open('neutral_atom_hybrid.pkl', 'rb') as file:
    results = pickle.load(file)
    

epsilons = [0.1, 0.4]
slacks = [0.2, 0.6, 1, 1.6, 2]
modes = {'Active':1}#, 'Extra Rounds':2}
hybrid_modes = {'Hybrid (' + r'$\epsilon:$' + ' %.1fms)'%(i):i * 10**6 for i in epsilons}

ds = [11]

# active, p, ls_basis, d, idle, rounds, orig_slack, c, eps

proc_data = {}
rounds = {}
for d in ds:
    proc_data[d] = {}
    rounds[d] = {}
    errs = []
    for mode, i in modes.items():
        proc_data[d][mode] = {}
        rounds[d][mode]= {}
        for slack in slacks:
            p = np.mean([val for key, val in results[0].items() if key[0] == 0 and key[-1] == 0 and key[3] == d and key[-3] == slack * 10**6])
            errs = [val for key, val in results[0].items() if key[0] == i and key[-1] == 0 and key[3] == d and key[-3] == slack * 10**6]
            proc_data[d][mode][slack] = p/np.mean(errs)
            # rounds[d][mode][slack] = np.mean([key[-4] for key in results[0].keys() if key[0] == i and key[-1] == 0 and key[3] == d and key[-3] == slack * 10**6])
        pass
    for mode, i in hybrid_modes.items():
        proc_data[d][mode] = {}
        rounds[d][mode]= {}
        for slack in slacks:
            p = np.mean([val for key, val in results[0].items() if key[0] == 0 and key[-1] == 0 and key[3] == d and key[-3] == slack * 10**6])
            errs = [val for key, val in results[0].items() if key[0] == 3 and key[-1] == i and key[3] == d and key[-3] == slack*10**6]
            proc_data[d][mode][slack] = p / np.mean(errs)
            # rounds[d][mode][slack] = np.max([key[-4] for key in results[0].keys() if key[0] == 3 and key[-1] == i and key[3] == d and key[-3] == slack*10**6])
        pass

fig, ax = plt.subplots(1, 1, constrained_layout=True, figsize=(19, 9))
df = pd.DataFrame(proc_data[ds[0]])
df.plot.bar(rot=0, ax=ax, legend=False, edgecolor='k')
set_hatches(ax, df)
# ax.set_yscale('log')
ax.grid(linestyle='--', axis='y')
# ax.legend(ncol=3)
ax.set_ylim(0.5)
ax.axhline(y=1, color='k', linestyle='--')
fs = 44
ax.set_ylabel('Reduction', fontsize=fs)
ax.set_xlabel(r'Initial Slack, $\tau$ (ms)', fontsize=fs)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.setp(ax.get_xticklabels(), fontsize=fs)
plt.setp(ax.get_yticklabels(), fontsize=fs)
# ax.legend(ncol=3, fontsize=fs)
ax.axhline(y=1, linestyle='--', color='k')
ax.set_title(r'$d=11$', fontsize=fs)
handles, labels = ax.get_legend_handles_labels()
handle_list, label_list = [], []
for handle, label in zip(handles, labels):
    if label not in label_list:
        handle_list.append(handle)
        label_list.append(label)
lgd = fig.legend(handle_list, label_list, ncol=3, loc='lower center', fancybox=False, shadow=True, fontsize=fs, bbox_to_anchor=(0.5,-0.17))
# df = pd.DataFrame(rounds)
# print(df[11])

## Slack due to cultivation

In [ ]:
systems = {'Google':[35, 42+50, 660], 'IBM':[50, 70, 1500]}
fs = 40
num_shots = 100_000
result = []
tstates = 100
retries = [10, 100]
remap = {10:r'$p=0.0005$', 100:r'$p=0.001$'}
slack_result = {system:{remap[r]:[] for r in retries} for system in systems.keys()}

for system in systems.keys():
    h, cx, readout = systems[system]
    for retry in retries:
        for _ in range(num_shots):
            surface_cycle = 2 * h + 4 * cx + readout
            color_cycle = 2 * h + 6 * cx + readout
            num_retries = np.random.randint(1, retry)
            color_time = (color_cycle * 5 + 1000) * num_retries
            slack = color_time % surface_cycle
            slack_result[system][remap[retry]].append(slack)
            pass

fig, ax = plt.subplots(1, 1, constrained_layout=True, figsize=(12, 7))

colors = ['lightblue', 'lightgreen'] 
hatches = ['///', 'xxx']
positions = {'Google':[1,2], 'IBM':[4,5]}

# Create patches for legend with both color and hatch and black edges
patches = [plt.Rectangle((0,0),1,1, facecolor=color, hatch=hatch, 
                        edgecolor='black', linewidth=1) 
          for color,hatch in zip(colors,hatches)]

for i, system in enumerate(systems.keys()):
    for j, (retry, label) in enumerate(remap.items()):
        data = slack_result[system][label]
        mean = np.mean(data)
        bp = ax.boxplot([data], positions=[positions[system][j]], 
                       patch_artist=True,
                       boxprops=dict(facecolor=colors[j], hatch=hatches[j], color='black', linewidth=3),
                       whiskerprops=dict(linewidth=3),
                       capprops=dict(linewidth=3),
                       medianprops=dict(color='red', linewidth=3),
                       flierprops=dict(marker='o', markersize=10),
                       meanprops=dict(marker='s', markerfacecolor='white', markeredgecolor='black', markersize=10),
                       showmeans=True,
                       widths=0.6)  # Increase box width here

plt.setp(ax.get_xticklabels(), fontsize=fs)
plt.setp(ax.get_yticklabels(), fontsize=fs)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylabel('Slack (ns)', fontsize=fs)
# ax.set_xlabel('Physical Error Rate', fontsize=fs)
ax.grid(linestyle='--', axis='y')

ax.set_xticks([1.5, 4.5])
ax.set_xticklabels(['Google', 'IBM'])

# Add legend with labels from remap
ax.legend(patches, remap.values(), fontsize=fs-4)


## Optional: IBM idling experiments

This will require runtime access on IBM quantum systems, we provide the code here in case the user wants to run these idling experiments themselves.

In [ ]:
# Install qiskit dependencies
%pip install qiskit==0.43.2 qiskit-aer==0.12.1 qiskit-ibm-provider==0.5.0 qiskit-ibmq-provider==0.20.2 qiskit-terra==0.24.1 qiskit-ibm-provider==0.5.0 qiskit-ibmq-provider==0.20.2

In [ ]:
from qiskit_ibm_provider import IBMProvider
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit.transpiler import PassManager, InstructionDurations
from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling
from qiskit.circuit.library import XGate, RZGate

# Replace this string with your IBM token: 
token = 'token-string'
IBMProvider.save_account(token=token, overwrite=True)
provider = IBMProvider(instance='ibm-q/open/main')

In [ ]:
# Change reps to 20/200 to get the results for the paper
reps = 20 # 200

def transDD(circ, backend, echo="X", echo_num=2, qubit_list=[]):
    """
    Args:
        circ: QuantumCircuit object
        backend (qiskit.providers.ibmq.IBMQBackend): Backend to transpile and schedule the
        circuits for. The numbering of the qubits in this backend should correspond to
        the numbering used in `self.links`.
        echo: gate sequence (expressed as a string) to be used on the qubits. Valid strings 
        are `'X'` and `'XZX'`.
        echo_num: Number of times to repeat the sequence for qubits.
    Returns:
        transpiled_circuit: As `self.circuit`, but with the circuits scheduled, transpiled and
        with dynamical decoupling added.
    """

    initial_layout = []
    initial_layout += [
        circ.qubits[q] for q in range(circ.num_qubits)
    ]

    # transpile to backend and schedule
    # circuit = transpile(
    #     circ, backend, initial_layout=initial_layout,scheduling_method="alap"
    # )
    circuit = circ

    #then dynamical decoupling if needed
    if echo_num:

        # set up the dd sequences
        spacings = []
        if echo == "X":
            dd_sequences = [XGate()] * echo_num
            spacings.append(None)
        elif echo == "XZX":
            dd_sequences = [XGate(), RZGate(np.pi), XGate()] * echo_num
            d = 1.0 / (2 * echo_num - 1 + 1)
            spacing = [d / 2] + ([0, d, d] * echo_num)[:-1] + [d / 2]
            for _ in range(2):
                spacing[0] += 1 - sum(spacing)
            spacings.append(spacing)
        else:
            dd_sequences.append(None)
            spacings.append(None)

        # add in the dd sequences
        durations = InstructionDurations().from_backend(backend)
        if dd_sequences[0]:
            if echo_num:
                if qubit_list == []:
                    qubits = circ.qubits
                else:
                    qubits = qubit_list
            else:
                qubits = None
            pm = PassManager([ALAPScheduleAnalysis(durations),
                  PadDynamicalDecoupling(durations, dd_sequences,qubits=qubits)])
            circuit = pm.run(circuit)

        # make sure delays are a multiple of 16 samples, while keeping the barriers
        # as aligned as possible
        total_delay = [{q: 0 for q in circuit.qubits} for _ in range(2)]
        for gate in circuit.data:
            if gate[0].name == "delay":
                q = gate[1][0]
                t = gate[0].params[0]
                total_delay[0][q] += t
                new_t = 16 * np.ceil((total_delay[0][q] - total_delay[1][q]) / 16)
                total_delay[1][q] += new_t
                gate[0].params[0] = new_t

        # transpile to backend and schedule again
        # circuit = transpile(circuit, backend, scheduling_method="alap")

    return circuit

def base_ckt() -> QuantumCircuit:
    circuit = QuantumCircuit(1, 1)

    circuit.sx(0)
    for _ in range(reps):
        circuit.x(0)
        circuit.sx(0) 
        circuit.x(0)
        circuit.sx(0) 
        circuit.x(0)
        # circuit.cx(0, 1)
    circuit.sxdg(0)
    circuit.measure(0, 0)
    return circuit

def passive_ckt(delay:int) -> QuantumCircuit:
    circuit = QuantumCircuit(1, 1)
    circuit.h(0)
    for _ in range(reps):
        circuit.x(0)
        circuit.sx(0) 
        circuit.x(0)
        circuit.sx(0) 
        circuit.x(0)
        # circuit.cx(0, 1)
    circuit.delay(delay, unit='ns')
    circuit.h(0)
    circuit.measure(0, 0)
    return circuit

def active_ckt(delay:int) -> QuantumCircuit:
    circuit = QuantumCircuit(1, 1)

    stages = reps
    _delay = int(delay / stages)
    total_delay = 0
    _fixed_delay = 4
    circuit.h(0)
    for _ in range(reps):
        circuit.x(0)
        circuit.sx(0) 
        circuit.x(0)
        circuit.sx(0) 
        circuit.x(0)
        # circuit.cx(0, 1)
        if total_delay < delay:
            circuit.delay(_delay, unit='ns')
            total_delay += _fixed_delay
    circuit.h(0)
    circuit.measure(0, 0)
    return circuit

def passive_cktn(delay:int, n:int=5) -> QuantumCircuit:
    circuit = QuantumCircuit(n, n)
    circuit.h(list(range(n)))
    for _ in range(reps):
        circuit.x(list(range(n)))
        circuit.sx(list(range(n))) 
        circuit.x(list(range(n)))
        circuit.sx(list(range(n))) 
        circuit.x(list(range(n)))
        # circuit.cx(0, 1)
    circuit.delay(delay, unit='ns')
    circuit.h(list(range(n)))
    circuit.measure(list(range(n)), list(range(n)))
    return circuit

def active_cktn(delay:int, n:int=5) -> QuantumCircuit:
    stages = reps
    _delay = int(delay / stages)
    total_delay = 0
    _fixed_delay = 4
    circuit = QuantumCircuit(n, n)
    circuit.h(list(range(n)))
    for _ in range(reps):
        circuit.x(list(range(n)))
        circuit.sx(list(range(n))) 
        circuit.x(list(range(n)))
        circuit.sx(list(range(n))) 
        circuit.x(list(range(n)))
        # circuit.cx(0, 1)
        if total_delay < delay:
            circuit.delay(_delay, unit='ns')
            total_delay += _fixed_delay
    circuit.h(list(range(n)))
    circuit.measure(list(range(n)), list(range(n)))
    return circuit

def get_circuits():
    base = base_ckt()
    passive_ckts = []
    active_ckts = []
    for delay in [800, 1600, 2400, 3200, 4000, 5600]:
        passive = passive_cktn(delay, n=20)
        active = active_cktn(delay, n=20)
        passive_ckts.append(passive)
        active_ckts.append(active)
    # active = active_ckt(1000)
    return passive_ckts + active_ckts

def run_circuits(backend_name:str, circuits:list, shots:int=100_000):
    backend = provider.get_backend(backend_name)
    transpiled_ckts = []
    for ckt in circuits:
        temp = transpile(ckt, backend=backend, scheduling_method='asap', optimization_level=0)
        temp = transDD(temp, backend=backend)
        transpiled_ckts.append(temp)
        pass
    job = backend.run(transpiled_ckts, dynamic=False, shots=shots)
    return job

circuits = get_circuits()
backends = ['ibm_brisbane']
jobid = None
for bend in backends:
    job = run_circuits(bend, circuits)
    jobid = job.job_id()
    print(bend, ' ', job)

In [4]:
def retrieve_results(job_id:str, delays:list, baseline_en:bool=True):
    # job = service.job(job_id)
    results = provider.retrieve_job(job_id).result()
    # results = job.result()
    passive = {delay:None for delay in delays}
    active = {delay:None for delay in delays}
    if baseline_en:
        baseline = results.get_counts()[0] #results[0].data.c.get_counts()
        idx = 1
    else:
        idx = 0
    for delay in delays:
        passive[delay] = results.get_counts()[idx] #results[idx].data.c.get_counts()
        idx += 1
    for delay in delays:
        active[delay] = results.get_counts()[idx] #results[idx].data.c.get_counts()
        idx += 1
    if baseline_en:
        return {'base':baseline, 'passive':passive, 'active':active}
    else:
        return {'passive':passive, 'active':active}
    
def fidelity_n(delays:list, results:dict, super:bool=True):
    shots = 100_000
    active = results['active']
    passive = results['passive']
    # results are strings of 0s and 1s
    # need to count number of 0s for every qubit
    nqubits = len(list(passive[delays[-1]].keys())[0])
    per_qfid = {'Active':{d:[] for d in delays}, 'Passive':{d:[] for d in delays}}
    for delay in delays:
        for qubit in range(nqubits):
            idx = nqubits - 1 - qubit
            counts = 0
            for key in passive[delay].keys():
                if key[idx] == '0':
                    counts += passive[delay][key]
                    pass
                pass
            per_qfid['Passive'][delay].append(counts / shots)
            counts = 0
            for key in active[delay].keys():
                if key[idx] == '0':
                    counts += active[delay][key]
                    pass
                pass
            per_qfid['Active'][delay].append(counts / shots)
            pass
        pass
    mean_ = {'Active':{d:np.mean(per_qfid['Active'][d]) for d in delays},
             'Passive':{d:np.mean(per_qfid['Passive'][d]) for d in delays}}
    std_ = {'Active':{d:np.std(per_qfid['Active'][d]) for d in delays},
             'Passive':{d:np.std(per_qfid['Passive'][d]) for d in delays}}
    return mean_, std_

In [ ]:
delays = [800, 1600, 2400, 3200, 4000, 5600]

results_20 = retrieve_results('<REPLACE WITH THE JOB ID YOU GET>', delays, baseline_en=False)
results_200 = retrieve_results('<REPLACE WITH THE JOB ID YOU GET>', delays, baseline_en=False)

In [ ]:
num_times = 6
fs = 56
fig, axs = plt.subplots(1, 2, constrained_layout=True, figsize=(30, 10))
for i, results in enumerate([results_20, results_200]):
    ax = axs[i]
    means, std = fidelity_n(delays, results=results)
    for policy in ['Passive', 'Active']:
        mean_vals = list(means[policy].values())
        std_vals = list(std[policy].values())
        ax.errorbar(delays[:num_times], mean_vals[:num_times], yerr=std_vals[:num_times], label=policy, markersize=25, capsize=20)
    ax.set_xticks(delays[:num_times])
    ax.set_xticklabels(['%.1f'%(d / 1000) for d in delays[:num_times]])
    ax.set_yticks(np.arange(0.4, 0.91, 0.1))
    ax.set_xlabel(r'Idling period, $t_p$ ($\mu$s)', fontsize=fs)
    ax.set_ylabel('Mean Fidelity', fontsize=fs)
    # ax.legend(ncol=1, fontsize=fs)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.setp(ax.get_xticklabels(), fontsize=fs)
    plt.setp(ax.get_yticklabels(), fontsize=fs)
    ax.grid(linestyle='--')
axs[0].set_title('N = %i'%(20), fontsize=fs)
axs[1].set_title('N = %i'%(200), fontsize=fs)
handles, labels = ax.get_legend_handles_labels()
handle_list, label_list = [], []
for handle, label in zip(handles, labels):
    if label not in label_list:
        handle_list.append(handle)
        label_list.append(label)
lgd = fig.legend(handle_list, label_list, ncol=4, loc='lower center', fancybox=False, shadow=True, fontsize=fs, bbox_to_anchor=(0.5,-0.18))